In [1]:
import sys, glob
sys.path.append("../src")
import pandas as pd, numpy as np, folium
from clean import haversine_km, remove_jumps

f = sorted(glob.glob("../data/raw/bus_positions_*.csv"))[-1]
pos = pd.read_csv(f).drop_duplicates(["vehicle_id", "timestamp"]).dropna(subset=["route_id"])
clean = pd.concat([remove_jumps(g) for _, g in pos.groupby("vehicle_id")])

In [2]:
CELL = 0.0015  # about 150 m squares
g = clean.groupby("vehicle_id")
span_km = haversine_km(g.lat.min(), g.lon.min(), g.lat.max(), g.lon.max())
keep = span_km[(span_km > 2) & (g.size() >= 20)].index
tr = clean[clean.vehicle_id.isin(keep)].copy()

tr["cell"] = list(zip((tr.lat / CELL).round().astype(int), (tr.lon / CELL).round().astype(int)))
cells = tr.groupby("vehicle_id")["cell"].apply(set)
true_route = tr.groupby("vehicle_id")["route_id"].agg(lambda s: s.mode().iat[0]).loc[cells.index]
print(len(cells), "moving buses,", true_route.nunique(), "true routes")

4927 moving buses, 1385 true routes


In [3]:
from scipy.sparse import csr_matrix
ids = cells.index
cell_id = {c: i for i, c in enumerate(set().union(*cells))}
rows = [r for r, s in enumerate(cells) for _ in s]
cols = [cell_id[c] for s in cells for c in s]
M = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(ids), len(cell_id)))

inter = (M @ M.T).toarray()
size = np.asarray(M.sum(axis=1)).ravel()
dist = 1 - inter / np.minimum.outer(size, size)

In [4]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import homogeneity_score, completeness_score

pred = DBSCAN(eps=0.3, min_samples=2, metric="precomputed").fit_predict(dist)
mask = pred >= 0
print(f"{pred.max() + 1} clusters | {mask.mean():.0%} of buses clustered")
print(f"homogeneity {homogeneity_score(true_route[mask], pred[mask]):.2f} | "
      f"completeness {completeness_score(true_route[mask], pred[mask]):.2f}")

278 clusters | 26% of buses clustered
homogeneity 0.72 | completeness 0.90


In [5]:
colors = ["red", "blue", "green", "purple", "orange", "darkred", "cadetblue", "darkgreen"]
top = pd.Series(pred[mask]).value_counts().index[:8]
m4 = folium.Map(location=[28.61, 77.21], zoom_start=11, tiles="OpenStreetMap")
for color, c in zip(colors, top):
    for vid in ids[pred == c]:
        t = tr[tr.vehicle_id == vid].sort_values("timestamp")
        folium.PolyLine(t[["lat", "lon"]].values.tolist(), color=color, weight=3, opacity=0.6).add_to(m4)
m4.save("../data/processed/routes_map.html")
m4

In [6]:
from sklearn.cluster import AgglomerativeClustering

for t in [0.3, 0.5, 0.7]:
    lab = AgglomerativeClustering(n_clusters=None, metric="precomputed",
                                  linkage="average", distance_threshold=t).fit_predict(dist)
    sizes = pd.Series(lab).value_counts()
    m = pd.Series(lab).map(sizes).values >= 2
    print(f"t={t}: {(sizes >= 2).sum()} clusters | {m.mean():.0%} clustered | "
          f"homogeneity {homogeneity_score(true_route[m], lab[m]):.2f} | "
          f"completeness {completeness_score(true_route[m], lab[m]):.2f}")

t=0.3: 438 clusters | 20% clustered | homogeneity 0.88 | completeness 0.90
t=0.5: 995 clusters | 69% clustered | homogeneity 0.84 | completeness 0.84
t=0.7: 540 clusters | 96% clustered | homogeneity 0.71 | completeness 0.82


In [ ]:
lab = AgglomerativeClustering(n_clusters=None, metric="precomputed",
                              linkage="average", distance_threshold=0.5).fit_predict(dist)
top = pd.Series(lab).value_counts().index[:8]
m5 = folium.Map(location=[28.61, 77.21], zoom_start=11, tiles="OpenStreetMap")
for color, c in zip(colors, top):
    for vid in ids[lab == c]:
        t = tr[tr.vehicle_id == vid].sort_values("timestamp")
        folium.PolyLine(t[["lat", "lon"]].values.tolist(), color=color, weight=3, opacity=0.6).add_to(m5)
m5.save("../data/processed/routes_map_avg.html")
m5